# H2-1 extension, step 4: type x tier cell sizes

**Purpose**: before the step 6 main regression, check whether each Type (A/B/C) x tier (3 tiers / 5 quintiles) combination has enough households.
The VIF check (done separately after step 3) passed, so once the sample sizes are confirmed we are ready for step 6.

**Decision rule**: for cells with fewer than 30 households, regression results for that combination are treated as reference only.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)


## (Rebuild) steps 1-3: same pipeline as before

In [2]:
tier_df = load_tiers()
panel = load_panel()
recipients = set(tier_df.index[tier_df["will_receive"] == 1])
print("Pipeline rebuilt. panel shape:", panel.shape)

Pipeline rebuilt. panel shape: (172500, 9)


## Step 4: type x tier cell-size table

For each Type (A/B/C), count the unique households that received that campaign type **at least once** in weeks 33-101, by tier.

In [3]:
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)

print("[Type x 3 tiers: recipient households (main analysis)]")
cell3 = cell_size_table("tier3")
print(cell3)

print()
print("[Type x 5 quintiles: recipient households (supplementary analysis)]")
cell5 = cell_size_table("tier5")
print(cell5)


[Type x 3 tiers: recipient households (main analysis)]
       TypeA  TypeB  TypeC
tier3                     
low      187     90     28
mid      554    305     74
high     772    628    295

[Type x 5 quintiles: recipient households (supplementary analysis)]
              TypeA  TypeB  TypeC
tier5                            
q1 (lowest)      74     42     16
q2              192     92     25
q3              328    178     40
q4              444    293     92
q5 (highest)    475    418    224


In [4]:
THRESH = 30
print(f"Decision rule: cell size below {THRESH} households -> reference only")
print()

long3 = cell3.stack().reset_index()
long3.columns = ["tier3", "Type", "n_households"]
below3 = long3[long3["n_households"] < THRESH]
print("[3-tier basis: cells below the threshold]")
print(below3.to_string(index=False) if len(below3) > 0 else "None (all combinations have 30 or more households)")

print()
long5 = cell5.stack().reset_index()
long5.columns = ["tier5", "Type", "n_households"]
below5 = long5[long5["n_households"] < THRESH]
print("[5-quintile basis: cells below the threshold]")
print(below5.to_string(index=False) if len(below5) > 0 else "None")


Decision rule: cell size below 30 households -> reference only

[3-tier basis: cells below the threshold]
tier3  Type  n_households
  low TypeC            28

[5-quintile basis: cells below the threshold]
      tier5  Type  n_households
q1 (lowest) TypeC            16
         q2 TypeC            25


**Summary**

- **3-tier basis**: only low x TypeC = **28 households** falls below the threshold (30), and only by 2, so it is borderline. The other 8 combinations all have 74 or more.
- **5-quintile basis**: quintile 1 x TypeC = 16 and quintile 2 x TypeC = 25 are below the threshold. This means the concern a teammate raised earlier,
  "too few households for TypeC x low tier", **is actually confirmed at the quintile level**.

-> This confirms with data that designing it as "3 tiers for the main analysis, quintiles as a supplementary analysis" was right.
On the 3-tier basis the gate is effectively passed (only low x TypeC is borderline); split into quintiles, TypeC cannot hold up.

### Supplementary check: campaign reach by tier

Check whether the example a teammate gave, "22.1% of quintile 1 received a campaign vs 96.1% of quintile 5", also shows up in the actual data.

In [5]:
tier_df["received_any"] = tier_df.index.isin(recipients)

reach3 = tier_df.groupby("tier3", observed=True)["received_any"].agg(["sum", "count"])
reach3["reach_pct"] = (reach3["sum"] / reach3["count"] * 100).round(1)
reach3.columns = ["n_recipients", "n_households", "reach_pct"]
print("[Campaign reach by tier (all types combined)]")
print(reach3)

print()
reach5 = tier_df.groupby("tier5", observed=True)["received_any"].agg(["sum", "count"])
reach5["reach_pct"] = (reach5["sum"] / reach5["count"] * 100).round(1)
reach5.columns = ["n_recipients", "n_households", "reach_pct"]
print("[Campaign reach by quintile (all types combined)]")
print(reach5)


[Campaign reach by tier (all types combined)]
       n_recipients  n_households  reach_pct
tier3                                       
low             210           834       25.2
mid             591           834       70.9
high            783           832       94.1

[Campaign reach by quintile (all types combined)]
              n_recipients  n_households  reach_pct
tier5                                              
q1 (lowest)             89           500       17.8
q2                     208           500       41.6
q3                     353           500       70.6
q4                     453           500       90.6
q5 (highest)           481           500       96.2


In [6]:
reach_by_type = pd.DataFrame({
    t: cell3[t] / tier_df.groupby("tier3", observed=True).size() * 100
    for t in ["TypeA", "TypeB", "TypeC"]
}).round(1)
print("[Reach by tier x type (%)]")
print(reach_by_type)


[Reach by tier x type (%)]
       TypeA  TypeB  TypeC
tier3                     
low     22.4   10.8    3.4
mid     66.4   36.6    8.9
high    92.8   75.5   35.5


**Summary**

- Reach across all types, 3 tiers: low **25.2%** vs high **94.1%**, close to the figures the teammate gave (22.1% vs 96.1%).
  **The data confirm that campaigns were not sent out at random but concentrated on high spenders.**
  -> This supports the teammate's concern that a plain "recipients vs non-recipients" comparison must not be used.
- By type, **TypeC has the strongest relative skew** (low 3.4% vs high 35.5%, about 10.4 times).
  That is much larger than TypeA (4.1 times) and TypeB (7 times). TypeC also has the smallest sample and the strongest targeting skew,
  so any conclusion about TypeC must be handled with extra care.

## Verdict: did we pass the step 4 gate?

| Item | Result | Verdict |
|---|---|---|
| 3-tier cell sizes | 8 of 9 combinations have 74 or more, 1 (low x TypeC) has 28 | **Pass** (one borderline cell, footnote it in conclusions) |
| 5-quintile cell sizes | 2 TypeC combinations below the threshold | Quintiles are used only as a supplementary analysis (as originally planned) |
| Selection-bias check | Reach of low vs high spenders is 25.2% vs 94.1%, very large | Reconfirms that household fixed effects must be used in step 6 |

**Next step**: we can proceed to step 5 (parallel pre-trend test). The low x TypeC combination and TypeC in general
should be interpreted in the step 6 results with the caveat "borderline sample".

In [7]:
# Low x TypeC recipient households (TypeC active at least once in weeks 33-101)
low_typeC_hh = panel.loc[
    (panel["active_TypeC"] == 1) & (panel["tier3"] == "low"), "household_key"
].unique()
print(f"Low x TypeC recipient households: {len(low_typeC_hh)}")

# Overlap with households that had zero transactions in the pre-period (weeks 17-32)
tier_df["was_zero_pre"] = (tier_df["avg_weekly_spend_pre"] == 0)
overlap = tier_df.loc[tier_df.index.isin(low_typeC_hh), "was_zero_pre"]
n_overlap = overlap.sum()
print(f"Of these, households with zero pre-period spend: {n_overlap} ({n_overlap/len(low_typeC_hh)*100:.1f}%)")

Low x TypeC recipient households: 28
Of these, households with zero pre-period spend: 2 (7.1%)


In [8]:
_t5 = dict(zip(["q1 (lowest)", "q2", "q3", "q4", "q5 (highest)"], ["q1 (lowest)", "q2", "q3", "q4", "q5 (highest)"]))  # internal labels are already the English names written to the CSVs
cell3.rename(index=TIER_EN).rename_axis("tier").reset_index().to_csv(OUT_DIR / "cell_size_3x3.csv", index=False)
cell5.rename(index=_t5).rename_axis("tier").reset_index().to_csv(OUT_DIR / "cell_size_5x3.csv", index=False)
reach3.rename(index=TIER_EN) \
    .rename_axis("tier").reset_index().to_csv(OUT_DIR / "reach_by_tier3.csv", index=False)
reach5.rename(index=_t5) \
    .rename_axis("tier").reset_index().to_csv(OUT_DIR / "reach_by_tier5.csv", index=False)
(cell3 / tier_df.groupby("tier3", observed=True).size().values[:, None] * 100).rename(index=TIER_EN) \
    .rename_axis("tier").reset_index().to_csv(OUT_DIR / "reach_by_type_tier3.csv", index=False)
pd.DataFrame([{
    "cell_size_threshold": THRESH, "n_cells_below_threshold_3x3": len(below3), "n_cells_below_threshold_5x3": len(below5),
    "low_typeC_households": len(low_typeC_hh), "low_typeC_zero_pre": int(n_overlap),
    "low_typeC_zero_pre_pct": n_overlap / len(low_typeC_hh) * 100,
}]).to_csv(OUT_DIR / "cell_size_checks.csv", index=False)